# BlockGeo-QEM quick start

This notebook runs the whole pipeline on a small problem that takes about a minute on a laptop CPU. It generates uniform transverse-field Ising (TFIM) circuits on a 4-qubit chain, measures them with a finite shot budget on a simulated noisy device, mitigates the two-qubit block states with a few baselines and with BlockGeo-QEM, and compares the errors. Data are written to a temporary directory, so nothing is added to the repository.

Run it from the `examples/` directory or from the repository root.

In [1]:
import os, sys, subprocess, tempfile
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
WORK = Path(tempfile.mkdtemp(prefix="blockgeo_quickstart_"))
os.environ["BLOCKGEO_WORKDIR"] = str(WORK)          # read by src/paths.py at import time
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")   # the CPU is enough here
sys.path.insert(0, str(REPO / "src"))
print("repository:", REPO.name, "| temporary working directory created")

repository: BlockGeo-QEM | temporary working directory created


## 1. Generate circuits and simulate them

`gen_v3.py` draws random-angle TFIM circuits, simulates them with the Pauli-twirled noise model of the default device chain, and stores the ideal block coordinates together with the outcome distributions of the nine covering measurement settings at noise scales 1, 3 and 5. Here we use 100 training and 20 test circuits.

In [2]:
cmd = [sys.executable, str(REPO / "src" / "gen_v3.py"), "--family", "tfim", "--n", "4", "--depths", "1",
       "--n_train", "100", "--n_test", "20", "--K", "3", "--workers", "2"]
print(subprocess.run(cmd, check=True, capture_output=True, text=True).stdout.strip())

tfim n=4 d=1 done in 12s


## 2. Finite-shot measurements

`run_v3.draws` samples the covering settings with $10^4$ shots per circuit, applies readout errors and readout-error mitigation, and estimates the 15 Pauli coordinates of every two-qubit block. The arrays have the shape (circuits, blocks, 15).

In [3]:
import numpy as np
import pandas as pd
import run_v3

D = run_v3.draws("tfim", 4, 1, budget=10000)
print("training estimates", D["tr1"].shape, "| test estimates", D["te1"].shape)

training estimates (100, 3, 15) | test estimates (20, 3, 15)


## 3. Mitigation

We compare the unmitigated estimates with a per-coordinate linear regression (OLS), a message-passing network on the qubit chain (GNN-QEM) and BlockGeo-QEM, all trained on the same measured training circuits and their ideal block coordinates.

In [4]:
import baselines as B
import baselines_v3 as B3
from v3 import BlockGeoQEM2

Y = D["tr"]["S_ideal"]
tr1, te1 = {1: D["tr1"]}, {1: D["te1"]}
cq, ce = B3.chain_calibration(4)
preds = {
    "Noisy": D["te1"],
    "OLS": B.ols_per_coord(tr1, Y, te1, (1,)),
    "GNN-QEM": B3.gnn_qem(D["tr1"], Y, D["te1"], cq, ce),
    "BlockGeo-QEM": BlockGeoQEM2(beta=1.0).fit(tr1, Y).predict(te1),
}

## 4. Errors

`local_bures` and `local_trace` are the Bures and trace distances between the mitigated and the ideal block states, averaged over the blocks of a circuit; `err_purity` is the error of the block purity. The table shows means over the 20 test circuits.

In [5]:
from bench import metrics

table = pd.concat([metrics(name, p, D["te"]["S_ideal"], D["te1"]) for name, p in preds.items()])
table.groupby("method", sort=False)[["local_bures", "local_trace", "err_purity"]].mean().round(4)

,local_bures,local_trace,err_purity
method,,,
Noisy,0.1427,0.0458,0.0458
OLS,0.1014,0.0344,0.0201
GNN-QEM,0.0579,0.0194,0.0094
BlockGeo-QEM,0.0186,0.0140,0.0042


## 5. Physical outputs

Coordinate-level methods can return block coordinates that do not belong to any density matrix. BlockGeo-QEM returns density matrices by construction. The cell below builds the matrix $(I + \sum_k s_k P_k)/4$ of every block without any projection and counts the blocks with an eigenvalue below $-10^{-6}$, well beyond floating-point round-off.

In [6]:
import qemlib as L

P = L.pauli_ops(2)   # the 15 two-qubit Pauli matrices

def fraction_unphysical(S):
    rho = (np.eye(4) + np.einsum("...k,kij->...ij", S.astype(complex), P)) / 4   # no projection
    return float(np.mean(np.linalg.eigvalsh(rho)[..., 0] < -1e-6))

pd.Series({name: fraction_unphysical(p) for name, p in preds.items()}, name="fraction of unphysical blocks")

Noisy           0.833333
OLS             0.900000
GNN-QEM         0.866667
BlockGeo-QEM    0.000000
Name: fraction of unphysical blocks, dtype: float64

The full studies, with all fourteen baselines, three circuit families, several chain lengths and shot budgets, are described in `docs/reproduce.md`.

In [7]:
import shutil
shutil.rmtree(WORK)   # remove the temporary data